In [1]:
import tensorflow as tf
# tf.config.set_visible_devices([], 'GPU')

import numpy as np
import pandas as pd
import pyterrier as pt
import os
import ir_datasets
from urllib.parse import urlparse, parse_qs
import re
from sklearn.model_selection import train_test_split
import copy

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from keras.layers import Input
import time
from sklearn.metrics.pairwise import cosine_similarity

from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import load_model

2025-01-20 12:30:15.959239: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-20 12:30:15.967317: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1737365415.977829   23998 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1737365415.980852   23998 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-20 12:30:15.991955: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
if not pt.started():
    pt.init()

/tmp/ipykernel_23998/3057724015.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_23998/3057724015.py:2: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [3]:
dataset = ir_datasets.load('istella22/test')

In [ ]:
for i in dataset.docs:
    print(i)
    break

[INFO] To use the Istella22 dataset, you must read and accept the Istella22 Licence Agreement, found here: <https://istella.ai/data/istella22-dataset/>
[INFO] If you have a local copy of https://www.istella.ai/dataset/istella22.tar.gz, you can symlink it here to avoid downloading it again: /home/ersel/.ir_datasets/downloads/c2e49dca9730fbb14164ed890756dc1d
[INFO] [starting] https://www.istella.ai/dataset/istella22.tar.gz
https://www.istella.ai/dataset/istella22.tar.gz: 2.3%| 606M/26.5G [01:51<1:19:23, 5.44MB/s] 

# CREATION OF WEBSITE GAS MODEL

In [ ]:

# takes toooooooo much time
if False:
    num_of_docs = len(dataset.docs)
    
    
    docno_to_index = {}
    erronious_indexes = []
    for index in range(num_of_docs):
        try:
            docno_to_index[dataset.docs[index].doc_id] = index
        except:
            erronious_indexes.append(index)
            print("error at:", index)
            continue

In [17]:
INDEX_EXISTS = True
index_dir = '/media/ersel/Expansion/istella22_index3'

if INDEX_EXISTS:
    index = pt.IndexFactory.of(index_dir)
else:
    def doc_to_dict_generator(docs):
        global error_no
        for doc in docs:
            try:
                yield {"docno": doc.doc_id, "text": doc.text}
            except:
                yield {"docno": str(error_no), "text": "istella document error"}
                error_no -= 1
    
    indexer = pt.index.IterDictIndexer(index_dir)
    index = indexer.index(doc_to_dict_generator(dataset.docs))


In [18]:
bm25_retriever = pt.BatchRetrieve(index, wmodel="BM25") % 100

/tmp/ipykernel_8401/3716249422.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_retriever = pt.BatchRetrieve(index, wmodel="BM25") % 100


In [19]:
search_res = bm25_retriever.search("sad life")
print(search_res)
print(len(search_res))

   qid    docid             docno  rank      score     query
0    1  2775598  1990011300499881     0  23.154382  sad life
1    1  1935575  1990010900176593     1  23.025193  sad life
2    1  3032524  1990011401260999     2  22.925853  sad life
3    1   406548  1990010102281031     3  22.835376  sad life
4    1  1021330  1990010401842904     4  22.354891  sad life
..  ..      ...               ...   ...        ...       ...
95   1   857857  1990010302557020    95  19.702936  sad life
96   1  1222075  1990010501697919    96  19.694342  sad life
97   1   716068  1990010300627743    97  19.647187  sad life
98   1  2889559  1990011302137931    98  19.636935  sad life
99   1  3323228  1990011502613031    99  19.631194  sad life

[100 rows x 6 columns]
100


Istella22Doc(doc_id='1990011300499881', title='A life story | Sad Video | Lover of Sadness', url='http://www.loverofsadness.net/sad_video.php?id=74&o=1', text="Lover of Sadness Contact | Suggestion | Claim Credit | Terms & Condition | Privacy Policy Copyright © all rights reserverd 2009 A Life Story Description: A life story of a guy and his loving family Sad Sad with esothori plzz can to laefi Sadness Life video &lt;&lt; Previous Video Music Video Next Video >> Nepeta says: 16 Nov, 2014 11:52 PM I cried at the dog part. Noooooooo!!! Dogg!!!!!! ;,( satyaranjan sahoo says: 30 Jan, 2015 01:29 PM it really heart touchinng..... Do not post other site's link, it will be considered as spam Umm, are you really just giving this info out for nointhg? Created by Bony Yousuf - From Gloomy Sunday My Account | Logout I miss u shona... Register | Login Tags: Life , Love , Death , Animal 2 Post a Comment Inspirational A life story Awesome!!! Separation Lost Love Sacrifice Wedding Suicide Missing Pare

In [12]:
queries = {}
relevant_set = set()
for query in dataset.queries:
    queries[query.query_id] = {"text": query.text, "docs":{}}
for qrel in dataset.qrels:
    queries[qrel.query_id]['docs'][qrel.doc_id] = qrel.relevance
    relevant_set.add(qrel.doc_id)


In [13]:
def dcg_at_k(relevances, k=None):
    if k:
        relevances = relevances[:k]
    return sum([rel / np.log2(i + 1) for i, rel in enumerate(relevances, 1)])

def ndcg_at_k(relevances, k=None):
    dcg = dcg_at_k(relevances, k)
    idcg = dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0

In [14]:
queries2 = {} 
for qid, q_dict in queries.items():
    # qid, q_dict = '263', queries['263']
    try:
        bm25_res = bm25_retriever.search(q_dict['text'])
    except:
        continue
    doc_dict = q_dict['docs']
    state_flag = False
    for doc in bm25_res.iloc:
        if doc.docno in doc_dict:
            state_flag = True
            break
    if state_flag:
        q_dict['bm25'] = bm25_res
        queries2[qid] = q_dict

# FEATURE FUNCTIONS

In [2]:
def title_query_term_overlap(doc, query):
    title_terms = set(doc.title.split())
    query_terms = set(query.split())
    return len(title_terms & query_terms)

def title_query_jaccard_similarity(doc, query):
    title_terms = set(doc.title.split())
    query_terms = set(query.split())
    intersection = len(title_terms & query_terms)
    union = len(title_terms | query_terms)
    return intersection / union if union else 0

def title_query_dice_similarity(doc, query):
    title_terms = set(doc.title.split())
    query_terms = set(query.split())
    intersection = len(title_terms & query_terms)
    return (2 * intersection) / (len(title_terms) + len(query_terms)) if title_terms and query_terms else 0

def title_query_position(doc, query):
    title_words = doc.title.split()
    query_terms = query.split()
    for i, word in enumerate(title_words):
        if word in query_terms:
            return i
    return -1

def exact_match_title_query(doc, query):
    return 1 if doc.title.strip().lower() == query.strip().lower() else 0

def query_length(query):
    return len(query.split())

def query_character_length(query):
    return len(query)

def term_overlap(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.text.split())
    return len(query_terms & doc_terms)

def jaccard_similarity(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.text.split())
    intersection = len(query_terms & doc_terms)
    union = len(query_terms | doc_terms)
    return intersection / union if union else 0

def dice_similarity(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.text.split())
    intersection = len(query_terms & doc_terms)
    return (2 * intersection) / (len(query_terms) + len(doc_terms)) if query_terms and doc_terms else 0

def term_overlap_extra(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.extra_text.split())
    return len(query_terms & doc_terms)

def jaccard_similarity_extra(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.extra_text.split())
    intersection = len(query_terms & doc_terms)
    union = len(query_terms | doc_terms)
    return intersection / union if union else 0

def dice_similarity_extra(doc, query):
    query_terms = set(query.split())
    doc_terms = set(doc.extra_text.split())
    intersection = len(query_terms & doc_terms)
    return (2 * intersection) / (len(query_terms) + len(doc_terms)) if query_terms and doc_terms else 0









def document_length(doc):
    return len(doc.text.split())

def document_character_length(doc):
    return len(doc.text)

def average_sentence_length(doc):
    sentences = re.split(r'[.!?]', doc.text)
    sentences = [sent.strip() for sent in sentences if sent.strip()]  # Remove empty sentences and leading/trailing spaces
    return sum(len(sent.split()) for sent in sentences) / len(sentences) if sentences else 0

def stopword_proportion(doc):
    words = doc.split()
    stopword_count = sum(1 for word in words if word.lower() in STOPWORDS)
    return stopword_count / len(words) if words else 0

def unique_word_count(doc):
    return len(set(doc.text.split()))


def url_depth(doc):
    return doc.url.count('/')

def has_query_parameters(doc):
    return '?' in doc.url

def get_features(doc, query):
    return [
        title_query_term_overlap(doc, query),
        title_query_jaccard_similarity(doc, query),
        title_query_dice_similarity(doc, query),
        title_query_position(doc, query),
        exact_match_title_query(doc, query),
        query_length(query),
        query_character_length(query),
        term_overlap(doc, query),
        jaccard_similarity(doc, query),
        dice_similarity(doc, query),
        term_overlap_extra(doc, query),
        jaccard_similarity_extra(doc, query),
        dice_similarity_extra(doc, query),
        document_length(doc),
        document_character_length(doc),
        average_sentence_length(doc),
        unique_word_count(doc),
        url_depth(doc),
        has_query_parameters(doc),
    ]

In [ ]:
for qid, q_dict in queries2.items():
    q_text = q_dict['text']
    q_dict['features'] = []
    for row in q_dict['bm25'].iloc:
        bm25_score = row['score']
        docindex = int(row['docid'])
        doc = dataset.docs[docindex]
        query = q_dict['text']
        qd_features = get_features(doc, query)
        qd_features.append(bm25_score)
        q_dict['features'].append(qd_features)

In [ ]:
for qid, q_dict in queries2.items():
    q_dict['scores'] = []
    for row in q_dict['bm25'].iloc:
        score = 0
        if row.docno in q_dict['docs']:
            score = q_dict['docs'][row.docno]
        q_dict['scores'].append(score)

# Shared Funcs

In [2]:
def dcg_at_k(relevances, k=None):
    if k:
        relevances = relevances[:k]
    return sum([rel / np.log2(i + 1) for i, rel in enumerate(relevances, 1)])

def ndcg_at_k(relevances, k=None):
    dcg = dcg_at_k(relevances, k)
    idcg = dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0

In [3]:
def normal_access(container, i1, i2):
    return container[i1][i2]

def csr_matrix_access(container, i1, i2):
    return container[(i1, i2)]

In [4]:
def kendalls_tau(f1_index, f2_index, features_list, access=normal_access):
    concordont_count = 0
    discordont_count = 0
    try:
        n = len(features_list)
    except:
        n = features_list.shape[0]
    for d1 in range(n):
        d1_f1 = access(features_list, d1, f1_index)
        d1_f2 = access(features_list, d1, f2_index)
        for d2 in range(d1+1, n):
            d2_f1 = access(features_list, d2, f1_index)
            d2_f2 = access(features_list, d2, f2_index)
            if (
                    (
                        (d1_f1 > d2_f1)
                        and
                        (d1_f2 > d2_f2)
                    ) or
                    (
                        (d1_f1 < d2_f1)
                        and
                        (d1_f2 < d2_f2)
                    )
            ):
                concordont_count += 1
            elif (
                    (
                        (d1_f1 > d2_f1)
                        and
                        (d1_f2 < d2_f2)
                    ) or
                    (
                        (d1_f1 < d2_f1)
                        and
                        (d1_f2 > d2_f2)
                    )
            ):
                discordont_count += 1
    return (concordont_count - discordont_count) / (n*(n-1)/2)

In [5]:
def get_sim_key(f1, f2):
    return str(min(f1, f2)) + "#" + str(max(f1, f2))

In [6]:
def get_best_feature(features, placeholder):
    best_feature = None
    best_feature_index = None
    for index, feature in features:
        if not best_feature or feature >= best_feature:
            best_feature = feature
            best_feature_index = index 
    return best_feature_index, best_feature

def get_best_max_k(features, similarity_dict, k=5):
    best_feature = None
    best_feature_index = None
    for index, feature in features:
        if not best_feature:
            best_feature = feature
            best_feature_index = index
            continue
        
        other_features = []
        for index2, feature2 in features:
            if index == index2:
                continue
            key = get_sim_key(index, index2)
            sim_val = similarity_dict[key]
            other_features.append(feature2 - sim_val)
        feature_score = feature + sum(sorted(other_features, reverse=True)[:k]) / k
        
        if feature_score >= best_feature:
            best_feature = feature_score
            best_feature_index = index
    return best_feature_index, best_feature

def get_best_average_change(features, similarity_dict):
    best_feature = None
    best_feature_index = None
    for index, feature in features:
        if not best_feature:
            best_feature = feature
            best_feature_index = index
            continue
        
        average_decrement = 0
        for index2, _ in features:
            if index == index2:
                continue
            key = get_sim_key(index, index2)
            sim_val = similarity_dict[key]
            average_decrement += sim_val
        average_decrement /= len(features) - 1
        
        feature_score = feature - average_decrement
        if feature_score >= best_feature:
            best_feature = feature_score
            best_feature_index = index
    return best_feature_index, best_feature


def update_features(features, best_feature_index, similarity_dict):
    for feature_tuple in features:
        feature_index, feature_score = feature_tuple
        if feature_index != best_feature_index:
            key = get_sim_key(feature_index, best_feature_index)
            sim_val = similarity_dict[key]
            feature_tuple[1] -= sim_val

def GAS(features, similarity_dict, get_best_func=get_best_feature):
    features2 = [[index, score] for index, score in enumerate(features)]
    n = len(features2)
    features_ordered = []
    for i in range(n-1):
        best_feature_index, best_feature_score = get_best_func(features2, similarity_dict)
        print(best_feature_index, best_feature_score)
        features_ordered.append(best_feature_index)
        update_features(features2, best_feature_index, similarity_dict)
        features2 = [feature_tuple for feature_tuple in features2 if feature_tuple[0] != best_feature_index]
    features_ordered.append(features2[0][0])
    return features_ordered

def GAS_changing_bests(features, similarity_dict, get_best_func=get_best_feature, get_best_func2=get_best_max_k, cutting_point=41):
    get_best_function = get_best_func 
    features2 = [[index, score] for index, score in enumerate(features)]
    n = len(features2)
    features_ordered = []
    for i in range(n-1):
        if i == cutting_point:
            get_best_function = get_best_func2
        best_feature_index, best_feature_score = get_best_function(features2, similarity_dict)
        print(best_feature_index, best_feature_score)
        features_ordered.append(best_feature_index)
        update_features(features2, best_feature_index, similarity_dict)
        features2 = [feature_tuple for feature_tuple in features2 if feature_tuple[0] != best_feature_index]
    features_ordered.append(features2[0][0])
    return features_ordered

# LAMBDAMART

In [7]:
import lightgbm as lgb
from sklearn.datasets import load_svmlight_file

In [8]:
#from rankeval.dataset import Dataset
#from rankeval.model import RTEnsemble
TEST_FILE = '/media/ersel/Expansion/code/CENG778-istella22_trials/lambdamart/data/test.monoT5.svm'
MODEL_FILE = '/media/ersel/Expansion/code/CENG778-istella22_trials/lambdamart/models/lambdamart.monoT5.lgb'

In [9]:
X, y, q = load_svmlight_file(TEST_FILE, query_id=True)

In [10]:
divide_point = 1074050

if divide_point is None:
    old_qid = q[0]
    qid_count = 1
    for index, qid in enumerate(q):
        if qid != old_qid:
            qid_count += 1
            old_qid = qid
            if qid_count == 1501:
                divide_point = index
                break

X_train = X[:divide_point]
X_test = X[divide_point:]
y_train = y[:divide_point]
y_test = y[divide_point:]
q_train = q[:divide_point]
q_test = q[divide_point:]
print(divide_point)

1074050


In [11]:
feature_count = X_train.shape[1]
train_query_count = 100 # not 1500 because it takes 10 hours just to 

In [12]:
qid_limits_train = []
old_qid = q_train[0]
start_index = 0
for index, qid in enumerate(q_train):
    if qid != old_qid:
        old_qid = qid
        qid_limits_train.append((start_index, index))
        start_index = index
qid_limits_train.append((start_index, index + 1))
qid_rel_count_train = [limits[1] - limits[0] for limits in qid_limits_train]

qid_limits_test = []
old_qid = q_test[0]
start_index = 0
for index, qid in enumerate(q_test):
    if qid != old_qid:
        old_qid = qid
        qid_limits_test.append((start_index, index))
        start_index = index
qid_limits_test.append((start_index, index + 1))
qid_rel_count_test = [limits[1] - limits[0] for limits in qid_limits_test]

In [30]:
print(len(q_train[0:999]))
print(len(q_train[999:1999]))


999
1000


In [13]:
print(len(set(q_train)))
print(len(set(q_test)))
print(len(set(qid_limits_train)))
print(len(set(qid_limits_test)))
divide_point

1500
698
1500
698


1074050

In [13]:
lgbm_lmart = lgb.Booster(model_file=MODEL_FILE)
start_time = time.time()
predictions = lgbm_lmart.predict(X_test)
time_took = time.time() - start_time
print(time_took)

9.608296632766724


In [16]:
print(X_test.shape)

(427654, 221)


In [22]:
def generate_qid_data():
    old_qid = q_test[0]
    pairs = []
    for qid, pred_val, true_val in zip(q_test, predictions, y_test):
        if qid != old_qid:
            yield old_qid, pairs
            old_qid = qid
            pairs = []
        pairs.append((pred_val, true_val))
    yield qid, pairs

In [23]:
average_ndcg = 0
num_of_qs = 0
for qid, pairs in generate_qid_data():
    num_of_qs += 1
    pairs = sorted(pairs, key=lambda x: x[0], reverse=True)
    pair_scores = [pair[1] for pair in pairs]
    average_ndcg += ndcg_at_k(pair_scores)
average_ndcg = average_ndcg/num_of_qs

In [24]:
print(average_ndcg)

0.8747334768005786


In [20]:
"""
start, end = qid_limits_train[0]
start_time = time.time()
kendalls_tau(1,2,X_train[start: end],access=csr_matrix_access)
print(time.time() - start_time)
"""

'\nstart, end = qid_limits_train[0]\nstart_time = time.time()\nkendalls_tau(1,2,X_train[start: end],access=csr_matrix_access)\nprint(time.time() - start_time)\n'

In [47]:
start, end = (0,999)
f1 = 1
f2 = 2
print([X_test[start:end,f1].toarray().flatten().tolist()])
start_time = time.time()
sim_score = cosine_similarity([X_test[start:end,f1].toarray().flatten().tolist()], [X_test[start:end,f2].toarray().flatten().tolist()])
print(X_test[start:end,f1].shape)
print('time:', time.time() - start_time)
print('sim_score', sim_score)
print(X_test[start:end,f1].shape)
print(X_test[:,5][(0,0)])
print(X_test[:,5][(3,0)])

print(X_test[(0,5)])
print(X_test[(3,5)])

[[10111.0, 10198.0, 10241.0, 10362.0, 1049.0, 10565.0, 10750.0, 18136.0, 18143.0, 18185.0, 18186.0, 18229.0, 18290.0, 18417.0, 18470.0, 18946.0, 19105.0, 1922.0, 20220.0, 219.0, 2764.0, 289.0, 3194.0, 3204.0, 36795.0, 3681.0, 4105.0, 4256.0, 4355.0, 5295.0, 6727.0, 950.0, 9953.0, 9969.0, 23276.0, 23430.0, 24011.0, 25027.0, 2091.0, 1702.0, 1706.0, 10401.0, 911.0, 910.0, 638.0, 781.0, 352.0, 3160.0, 478.0, 11005.0, 11063.0, 11392.0, 11526.0, 11777.0, 11905.0, 11908.0, 11940.0, 12005.0, 12072.0, 12076.0, 12267.0, 12495.0, 12543.0, 12693.0, 12697.0, 12767.0, 13113.0, 13121.0, 13130.0, 13223.0, 13238.0, 13318.0, 13321.0, 13338.0, 13397.0, 13403.0, 13453.0, 13454.0, 13466.0, 13512.0, 13578.0, 1359.0, 13637.0, 13829.0, 13831.0, 13895.0, 13933.0, 13961.0, 14017.0, 14043.0, 14152.0, 14190.0, 14266.0, 14710.0, 14733.0, 14771.0, 14774.0, 14784.0, 14794.0, 14831.0, 14889.0, 15081.0, 15135.0, 15159.0, 15240.0, 15251.0, 15254.0, 15275.0, 15308.0, 15314.0, 15317.0, 15346.0, 15366.0, 15400.0, 15413.0,

In [13]:
similarity_dict = {
    get_sim_key(f1_index, f2_index): 0
    for f1_index in range(feature_count)
    for f2_index in range(f1_index + 1, feature_count)
}



start_time = 0
for i in range(train_query_count):
    start, end = qid_limits_train[i]
    print(i, 'start,end:', start, end, 'time:', time.time() - start_time)
    start_time = time.time()
    for f1_index in range(feature_count):
        for f2_index in range(f1_index + 1, feature_count):
            key = get_sim_key(f1_index, f2_index)
            f1_container = X_train[start:end, f1_index].toarray().flatten().tolist()
            f2_container = X_train[start:end, f2_index].toarray().flatten().tolist()
            score = cosine_similarity([f1_container], [f2_container])[0][0]
            similarity_dict[key] += score

for key in similarity_dict:
    similarity_dict[key] /= train_query_count

0 start,end: 0 999 time: 1737317560.5413291
1 start,end: 999 1999 time: 24.727888345718384
2 start,end: 1999 2406 time: 24.671825170516968
3 start,end: 2406 3406 time: 15.112207412719727
4 start,end: 3406 4406 time: 24.701514720916748
5 start,end: 4406 5406 time: 24.75353717803955
6 start,end: 5406 6406 time: 24.569411993026733
7 start,end: 6406 7406 time: 24.64017367362976
8 start,end: 7406 8406 time: 24.65135908126831
9 start,end: 8406 9406 time: 24.676490545272827
10 start,end: 9406 10406 time: 24.592056274414062
11 start,end: 10406 11406 time: 24.775387048721313
12 start,end: 11406 12406 time: 24.67381501197815
13 start,end: 12406 12683 time: 24.50228714942932
14 start,end: 12683 13125 time: 13.212754487991333
15 start,end: 13125 14125 time: 15.635936737060547
16 start,end: 14125 15125 time: 24.711149215698242
17 start,end: 15125 16125 time: 24.625218152999878
18 start,end: 16125 17125 time: 24.613009452819824
19 start,end: 17125 18124 time: 24.46829128265381
20 start,end: 18124 19

In [89]:
print(len(X_train[start:end, 5].toarray().flatten().tolist()))
print(X_train.shape)
print(y_train.shape)
len(y_train[start:end].flatten().tolist())
len(list(zip(f_container,scores)))

999
(1074050, 221)
(1074050,)


999

In [14]:
feature_scores = [0 for i in range(feature_count)]
for i in range(train_query_count):
    start, end = qid_limits_train[i]
    print(i, 'start,end:', start, end, 'time:', time.time() - start_time)
    start_time = time.time()
    scores = y_train[start:end].flatten().tolist()
    for f_index in range(feature_count):
        f_container = X_train[start:end, f_index].toarray().flatten().tolist()
        c_feature_list = list(zip(f_container, scores))
        feature_sorted_list = [second for _, second in sorted(c_feature_list, key=lambda x: x[0])]
        ndcg_score = ndcg_at_k(feature_sorted_list)
        feature_sorted_list.reverse()
        ndcg_score2 = ndcg_at_k(feature_sorted_list)
        feature_scores[f_index] += max(ndcg_score, ndcg_score2)

for i in range(feature_count):
    feature_scores[i] /= train_query_count

0 start,end: 0 999 time: 24.72922158241272
1 start,end: 999 1999 time: 1.7686898708343506
2 start,end: 1999 2406 time: 1.7659258842468262
3 start,end: 2406 3406 time: 0.7202715873718262
4 start,end: 3406 4406 time: 1.7665534019470215
5 start,end: 4406 5406 time: 1.7687337398529053
6 start,end: 5406 6406 time: 1.7793443202972412
7 start,end: 6406 7406 time: 1.7828772068023682
8 start,end: 7406 8406 time: 1.7696285247802734
9 start,end: 8406 9406 time: 1.7757833003997803
10 start,end: 9406 10406 time: 1.7731208801269531
11 start,end: 10406 11406 time: 1.7719438076019287
12 start,end: 11406 12406 time: 1.7743096351623535
13 start,end: 12406 12683 time: 1.7690463066101074
14 start,end: 12683 13125 time: 0.49458909034729004
15 start,end: 13125 14125 time: 0.7883274555206299
16 start,end: 14125 15125 time: 1.7769861221313477
17 start,end: 15125 16125 time: 1.774507761001587
18 start,end: 16125 17125 time: 1.7706575393676758
19 start,end: 17125 18124 time: 1.7735869884490967
20 start,end: 181

In [15]:
print(similarity_dict)
print(feature_scores)

{'0#1': np.float64(0.0), '0#2': np.float64(0.0), '0#3': np.float64(0.0), '0#4': np.float64(0.0), '0#5': np.float64(0.0), '0#6': np.float64(0.0), '0#7': np.float64(0.0), '0#8': np.float64(0.0), '0#9': np.float64(0.0), '0#10': np.float64(0.0), '0#11': np.float64(0.0), '0#12': np.float64(0.0), '0#13': np.float64(0.0), '0#14': np.float64(0.0), '0#15': np.float64(0.0), '0#16': np.float64(0.0), '0#17': np.float64(0.0), '0#18': np.float64(0.0), '0#19': np.float64(0.0), '0#20': np.float64(0.0), '0#21': np.float64(0.0), '0#22': np.float64(0.0), '0#23': np.float64(0.0), '0#24': np.float64(0.0), '0#25': np.float64(0.0), '0#26': np.float64(0.0), '0#27': np.float64(0.0), '0#28': np.float64(0.0), '0#29': np.float64(0.0), '0#30': np.float64(0.0), '0#31': np.float64(0.0), '0#32': np.float64(0.0), '0#33': np.float64(0.0), '0#34': np.float64(0.0), '0#35': np.float64(0.0), '0#36': np.float64(0.0), '0#37': np.float64(0.0), '0#38': np.float64(0.0), '0#39': np.float64(0.0), '0#40': np.float64(0.0), '0#41': 

In [23]:
features_ordered1 = GAS(feature_scores, similarity_dict, get_best_func=get_best_feature)
features_ordered2 = GAS(feature_scores, similarity_dict, get_best_func=get_best_max_k)
features_ordered3 = GAS(feature_scores, similarity_dict, get_best_func=get_best_average_change)
features_ordered4 = GAS_changing_bests(feature_scores, similarity_dict, get_best_func=get_best_average_change, get_best_func2=get_best_max_k)
features_ordered = features_ordered1

186 0.8874402923140855
220 0.43029541042461517
79 0.2761384962429653
0 0.24884516161012443
193 0.2469960687226768
218 0.23913442979123814
212 0.23913442979123814
205 0.23913442979123814
204 0.23913442979123814
202 0.23913442979123814
197 0.23913442979123814
94 0.23913442979123814
93 0.23913442979123814
91 0.23913442979123814
90 0.23913442979123814
89 0.23913442979123814
88 0.23913442979123814
87 0.23913442979123814
86 0.23913442979123814
85 0.23913442979123814
84 0.23913442979123814
83 0.23913442979123814
82 0.23913442979123814
81 0.23913442979123814
76 0.23913442979123814
62 0.23913442979123814
48 0.23913442979123814
174 0.23876173731309228
20 0.2377687175316899
104 0.23338400984135513
118 0.22188713957129108
200 0.21824433325134554
208 0.21430323021657027
107 0.18832539827877703
34 0.12883496954548676
217 0.11257180858132312
209 0.10966923013785926
103 0.054673743939152634
191 -0.02185038375847463
65 -0.07947705067008155
173 -0.09704759945120911
51 -0.19499911043819873
3 -0.200258265

In [24]:
print(features_ordered1)
print(features_ordered2)
print(features_ordered3)
print(features_ordered4)

[186, 220, 79, 0, 193, 218, 212, 205, 204, 202, 197, 94, 93, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 76, 62, 48, 174, 20, 104, 118, 200, 208, 107, 34, 217, 209, 103, 191, 65, 173, 51, 3, 74, 116, 219, 132, 99, 207, 214, 121, 160, 102, 172, 71, 32, 146, 203, 98, 166, 57, 117, 75, 101, 131, 37, 198, 70, 100, 46, 194, 18, 105, 211, 68, 96, 168, 56, 72, 112, 95, 19, 33, 108, 80, 47, 130, 97, 77, 177, 2, 60, 73, 113, 159, 67, 43, 1, 69, 110, 158, 61, 176, 54, 145, 42, 124, 58, 114, 144, 22, 45, 59, 126, 115, 63, 196, 40, 66, 12, 119, 55, 152, 122, 53, 44, 183, 109, 52, 135, 111, 14, 41, 29, 154, 39, 6, 134, 49, 171, 31, 179, 36, 28, 181, 190, 9, 138, 26, 184, 30, 213, 201, 25, 162, 140, 178, 38, 167, 27, 210, 35, 192, 120, 5, 23, 148, 50, 163, 189, 187, 149, 216, 170, 129, 165, 125, 136, 199, 4, 155, 123, 64, 128, 11, 151, 15, 17, 16, 127, 141, 157, 185, 156, 215, 169, 188, 182, 180, 206, 13, 24, 78, 106, 137, 21, 153, 164, 8, 175, 161, 133, 92, 10, 7, 147, 195, 142, 143, 150, 139]
[220

In [13]:
features_ordered1 = [186, 220, 79, 0, 193, 218, 212, 205, 204, 202, 197, 94, 93, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 76, 62, 48, 174, 20, 104, 118, 200, 208, 107, 34, 217, 209, 103, 191, 65, 173, 51, 3, 74, 116, 219, 132, 99, 207, 214, 121, 160, 102, 172, 71, 32, 146, 203, 98, 166, 57, 117, 75, 101, 131, 37, 198, 70, 100, 46, 194, 18, 105, 211, 68, 96, 168, 56, 72, 112, 95, 19, 33, 108, 80, 47, 130, 97, 77, 177, 2, 60, 73, 113, 159, 67, 43, 1, 69, 110, 158, 61, 176, 54, 145, 42, 124, 58, 114, 144, 22, 45, 59, 126, 115, 63, 196, 40, 66, 12, 119, 55, 152, 122, 53, 44, 183, 109, 52, 135, 111, 14, 41, 29, 154, 39, 6, 134, 49, 171, 31, 179, 36, 28, 181, 190, 9, 138, 26, 184, 30, 213, 201, 25, 162, 140, 178, 38, 167, 27, 210, 35, 192, 120, 5, 23, 148, 50, 163, 189, 187, 149, 216, 170, 129, 165, 125, 136, 199, 4, 155, 123, 64, 128, 11, 151, 15, 17, 16, 127, 141, 157, 185, 156, 215, 169, 188, 182, 180, 206, 13, 24, 78, 106, 137, 21, 153, 164, 8, 175, 161, 133, 92, 10, 7, 147, 195, 142, 143, 150, 139]
features_ordered2 = [220, 79, 193, 186, 218, 212, 205, 204, 202, 197, 94, 93, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 76, 62, 48, 20, 174, 104, 200, 118, 208, 107, 34, 0, 209, 217, 191, 65, 173, 103, 51, 3, 116, 219, 75, 132, 1, 207, 214, 160, 121, 102, 2, 71, 146, 203, 99, 57, 172, 32, 74, 117, 98, 37, 166, 70, 101, 4, 194, 5, 100, 6, 68, 131, 105, 96, 46, 95, 18, 108, 72, 7, 8, 9, 97, 73, 19, 77, 10, 11, 12, 80, 67, 13, 14, 15, 69, 130, 16, 198, 56, 17, 159, 112, 33, 21, 22, 23, 60, 145, 24, 25, 26, 27, 28, 29, 30, 31, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 47, 49, 158, 50, 52, 53, 54, 55, 58, 59, 61, 63, 144, 64, 66, 78, 92, 106, 109, 110, 111, 113, 114, 115, 119, 120, 122, 123, 124, 125, 126, 127, 128, 129, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 161, 162, 163, 164, 165, 167, 168, 169, 170, 171, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 187, 188, 189, 190, 192, 195, 196, 199, 201, 206, 210, 211, 213, 215, 216]
features_ordered3 = [220, 79, 193, 0, 218, 212, 205, 204, 202, 197, 94, 93, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 76, 62, 48, 174, 20, 104, 200, 208, 118, 107, 209, 34, 217, 1, 103, 191, 65, 3, 51, 74, 116, 219, 132, 207, 102, 173, 214, 121, 160, 99, 71, 32, 203, 146, 172, 98, 57, 2, 75, 117, 101, 166, 37, 194, 100, 70, 46, 131, 105, 211, 68, 198, 96, 18, 56, 72, 95, 168, 112, 33, 77, 108, 130, 97, 47, 73, 177, 19, 67, 60, 113, 80, 159, 43, 69, 110, 158, 61, 176, 145, 54, 42, 124, 58, 144, 114, 22, 45, 59, 126, 115, 196, 63, 40, 66, 12, 119, 55, 152, 122, 53, 44, 183, 109, 135, 52, 111, 14, 41, 29, 154, 39, 6, 134, 49, 171, 31, 179, 36, 28, 181, 190, 9, 138, 26, 184, 213, 30, 201, 162, 25, 140, 178, 38, 167, 27, 210, 35, 192, 120, 5, 23, 148, 50, 163, 189, 187, 149, 216, 170, 129, 165, 125, 136, 4, 199, 155, 123, 64, 128, 11, 151, 15, 17, 16, 7, 8, 141, 10, 13, 21, 24, 78, 127, 92, 106, 133, 137, 215, 157, 188, 185, 156, 186, 169, 182, 180, 206, 153, 164, 175, 161, 147, 195, 139, 142, 143, 150]
features_ordered4 = [220, 79, 193, 0, 218, 212, 205, 204, 202, 197, 94, 93, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 76, 62, 48, 174, 20, 104, 200, 208, 118, 107, 209, 34, 217, 1, 103, 191, 65, 3, 51, 116, 207, 219, 75, 2, 132, 173, 214, 121, 160, 102, 71, 57, 203, 146, 32, 172, 99, 117, 74, 166, 98, 37, 4, 70, 101, 5, 6, 194, 100, 131, 68, 46, 105, 7, 96, 95, 18, 108, 72, 8, 9, 97, 73, 10, 19, 77, 11, 12, 80, 13, 14, 67, 15, 16, 69, 130, 17, 198, 56, 159, 21, 22, 23, 112, 60, 24, 25, 26, 27, 28, 29, 30, 31, 33, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 47, 49, 158, 145, 50, 52, 53, 54, 55, 58, 59, 61, 63, 144, 64, 66, 78, 92, 106, 109, 110, 111, 113, 114, 115, 119, 120, 122, 123, 124, 125, 126, 127, 128, 129, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 161, 162, 163, 164, 165, 167, 168, 169, 170, 171, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 192, 195, 196, 199, 201, 206, 210, 211, 213, 215, 216]

In [14]:
def get_model(num_of_features, dimensions=[32, 32, 4]):
    layers = [Input(shape=(num_of_features,))]
    for dimension in dimensions:
        layers.append(Dense(dimension, activation='relu'))
    layers.append(Dense(1))
    model = Sequential(layers)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

In [15]:
def get_trained_lambdamart_model(x, y):
    params = {
        'objective': 'lambdarank',   # Set objective for LambdaMART
        'metric': 'ndcg',            # Evaluation metric
        'boosting_type': 'gbdt',     # Gradient boosting
        'num_leaves': 31,            # Max number of leaves per tree
        'learning_rate': 0.05,       # Learning rate
        'verbose': 0,                # Verbosity
        'max_depth': -1              # No max depth limit
    }
    train_data = lgb.Dataset(x.toarray(), label=y, group=qid_rel_count_train)
    model = lgb.train(params, train_data, num_boost_round=100)
    return model

In [18]:
class DynamicFeatureDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, x, y, batch_size, shuffle=True):
        self.num_samples, self.num_features = x.shape  # Total number of samples
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(self.num_samples)
        self.x = x
        self.y = y
        # self.indexes = range(self.num_samples)
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __len__(self):
        return int(np.floor(self.num_samples / self.batch_size))
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        
        # Generate or fetch the feature and label data for the batch
        batch_X = self.x[index * batch_size: index * batch_size + batch_size].toarray()
        batch_y = self.y[index * batch_size: index * batch_size + batch_size]
        
        return batch_X, batch_y

class DynamicFeatureDataTESTGenerator(tf.keras.utils.Sequence):
    def __init__(self, x, batch_size, shuffle=True):
        self.num_samples, self.num_features = x.shape  # Total number of samples
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(self.num_samples)
        self.x = x
        # self.indexes = range(self.num_samples)
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __len__(self):
        return int(np.floor(self.num_samples / self.batch_size))
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __getitem__(self, index):
        batch_X = self.x[index * batch_size: index * batch_size + batch_size].toarray()
        return batch_X
    



def pair_data_generator(x, y, batch_size=64):
    ending = x.shape[0]
    index = 0
    while(index<ending):
        e_index = index + batch_size
        if e_index > ending:
            e_index = ending
        x_batch = x[index: e_index].toarray()
        y_batch = y[index: e_index]
        # Yield the batch (features, labels)

        yield x_batch, y_batch
        index += batch_size

def data_generator(x, batch_size=64):
    ending = x.shape[0]
    index = 0
    while(index<ending):
        e_index = index + batch_size
        if e_index > ending:
            e_index = ending
        x_batch = x[index: e_index].toarray()
        
        # Yield the batch (features, labels)
        yield x_batch
        index += batch_size



# ds_counter = tf.data.Dataset.from_generator(count, args=[25], output_types=tf.int32, output_shapes = (), )
print(type(X_train))
print(X_train.shape)
print(type(X_train[(0,0)]))

aaa=X_train[:1000]
aaa=aaa.toarray()
aaa.shape
print(type(X_train[:10].toarray()))
print(type(y_train[:10]))
print()
for i,j in pair_data_generator(X_train,y_train):
    print(type(i))
    print(type(j))
    break

<class 'scipy.sparse._csr.csr_matrix'>
(1074050, 221)
<class 'numpy.float64'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [ ]:
best_score = -9999999999999999
best_model = None
best_num = None
batch_size = 64

iteration_time = time.time()
# for num_of_features in range(2,feature_count):
for num_of_features in range(feature_count - 1 , 1, -1):
    feature_indexes = sorted(copy.deepcopy(features_ordered[:num_of_features]))
    training_X = X_train[:, features_ordered[:num_of_features]]
    testing_X = X_test[:, features_ordered[:num_of_features]]
    
    # training_X = np.array(training_X)
    # testing_X = np.array(testing_X)

    # model = get_model(num_of_features)


    """
    generated_dataset = tf.data.Dataset.from_generator(
        lambda: pair_data_generator(training_X, y_train, batch_size=64),
        # pair_data_generator(training_X, y_train, batch_size=64),
        output_signature=(
            tf.TensorSpec(shape=(None, num_of_features), dtype=tf.float32),  # Features shape
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)   # Labels shape
        )
    )
    generated_dataset = generated_dataset.shuffle(buffer_size=1000).prefetch(tf.data.AUTOTUNE)
    """

    # model.fit(generated_dataset, epochs=100, steps_per_epoch=training_X.shape[0] // 64)
    # model.fit(generated_dataset, epochs=100, verbose=1)
    # generator = DynamicFeatureDataGenerator(training_X, y_train, batch_size)
    # model.fit(generator, epochs=10, verbose=1)
    # model.fit(training_X, y_train, epochs=100, batch_size=64, verbose=1)
    model = get_trained_lambdamart_model(training_X, y_train)
    print("iteration_took:", time.time() - iteration_time)
    iteration_time = time.time()
        
    # test_generator = DynamicFeatureDataTESTGenerator(testing_X, batch_size)
    # y_pred = model.predict(test_generator, verbose=0)
    # y_pred = model.predict(testing_X, verbose=0)

    start_time = time.time()
    y_pred = model.predict(testing_X.toarray())
    time_took = time.time() - start_time

    ndcg_average = 0
    for start, end in qid_limits_test:
        pred_ys = y_pred[start:end]
        real_ys = y_test[start:end]
        scored_docs = list(zip(pred_ys, real_ys))
        sorted_relevances = [data[1] for data in sorted(scored_docs, key=lambda x: x[0], reverse=True)]
        ndcg_score = ndcg_at_k(sorted_relevances)
        ndcg_average += ndcg_score
    ndcg_average /= len(qid_limits_test)
    score = ndcg_average
    print("feature_count:", num_of_features,"ndcg_score:", ndcg_average, "prediction_time:", time_took)

    if score > best_score:
        best_score = score
        best_model = model
        best_num = num_of_features

In [20]:

best_model.save_model('lambdamart_221_model.txt')

In [16]:
for features_ordered, model_name_suffix in [
    # (features_ordered1, "standard"),
    # (features_ordered2, "max5"),
    # (features_ordered3, "best_average"),
    (features_ordered4, "combined"),
]:
    best_score = -9999999999999999
    best_model = None
    best_num = None
    batch_size = 64
    
    iteration_time = time.time()
    # for num_of_features in range(2,feature_count):
    for num_of_features in range(feature_count - 1 , 1, -1):
        feature_indexes = sorted(copy.deepcopy(features_ordered[:num_of_features]))
        training_X = X_train[:, features_ordered[:num_of_features]]
        testing_X = X_test[:, features_ordered[:num_of_features]]
        
        # training_X = np.array(training_X)
        # testing_X = np.array(testing_X)
    
        # model = get_model(num_of_features)
    
    
        """
        generated_dataset = tf.data.Dataset.from_generator(
            lambda: pair_data_generator(training_X, y_train, batch_size=64),
            # pair_data_generator(training_X, y_train, batch_size=64),
            output_signature=(
                tf.TensorSpec(shape=(None, num_of_features), dtype=tf.float32),  # Features shape
                tf.TensorSpec(shape=(None, 1), dtype=tf.float32)   # Labels shape
            )
        )
        generated_dataset = generated_dataset.shuffle(buffer_size=1000).prefetch(tf.data.AUTOTUNE)
        """
    
        # model.fit(generated_dataset, epochs=100, steps_per_epoch=training_X.shape[0] // 64)
        # model.fit(generated_dataset, epochs=100, verbose=1)
        # generator = DynamicFeatureDataGenerator(training_X, y_train, batch_size)
        # model.fit(generator, epochs=10, verbose=1)
        # model.fit(training_X, y_train, epochs=100, batch_size=64, verbose=1)
        model = get_trained_lambdamart_model(training_X, y_train)
        print(f"{model_name_suffix}-{num_of_features} ### iteration_took:", time.time() - iteration_time)
        iteration_time = time.time()
            
        # test_generator = DynamicFeatureDataTESTGenerator(testing_X, batch_size)
        # y_pred = model.predict(test_generator, verbose=0)
        # y_pred = model.predict(testing_X, verbose=0)
    
        start_time = time.time()
        y_pred = model.predict(testing_X.toarray())
        time_took = time.time() - start_time
    
        ndcg_average = 0
        for start, end in qid_limits_test:
            pred_ys = y_pred[start:end]
            real_ys = y_test[start:end]
            scored_docs = list(zip(pred_ys, real_ys))
            sorted_relevances = [data[1] for data in sorted(scored_docs, key=lambda x: x[0], reverse=True)]
            ndcg_score = ndcg_at_k(sorted_relevances)
            ndcg_average += ndcg_score
        ndcg_average /= len(qid_limits_test)
        score = ndcg_average
        print("feature_count:", num_of_features,"ndcg_score:", ndcg_average, "prediction_time:", time_took)
    
        if score > best_score:
            best_score = score
            best_model = model
            best_num = num_of_features
        model.save_model(f'./models/gas_lambdamart_{model_name_suffix}_{num_of_features}_model.txt')

    print()
    print()
    print()


combined-220 ### iteration_took: 11.804270505905151
feature_count: 220 ndcg_score: 0.8570926822996079 prediction_time: 0.6521759033203125
combined-219 ### iteration_took: 12.27806043624878
feature_count: 219 ndcg_score: 0.8567498448934235 prediction_time: 0.5407507419586182
combined-218 ### iteration_took: 11.533017635345459
feature_count: 218 ndcg_score: 0.8553872946963876 prediction_time: 0.6428792476654053
combined-217 ### iteration_took: 11.399815797805786
feature_count: 217 ndcg_score: 0.8559933409847402 prediction_time: 0.6233270168304443
combined-216 ### iteration_took: 11.107233047485352
feature_count: 216 ndcg_score: 0.8573671486095582 prediction_time: 0.5929839611053467
combined-215 ### iteration_took: 11.524063110351562
feature_count: 215 ndcg_score: 0.8553491429563251 prediction_time: 0.5354673862457275
combined-214 ### iteration_took: 10.353582620620728
feature_count: 214 ndcg_score: 0.8552180195346186 prediction_time: 0.48485684394836426
combined-213 ### iteration_took: 9

2198

In [77]:
x = {}
c=0
for i in y_test:
    x.setdefault(i, 0)
    x[i] += 1
x

{np.float64(0.0): 1491011,
 np.float64(3.0): 2573,
 np.float64(4.0): 1040,
 np.float64(1.0): 6070,
 np.float64(2.0): 1010}

In [56]:
len(dataset.qrels)

10693

In [20]:
np.arange(10)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [19]:
np.random.rand(2, 4) 

array([[0.56969412, 0.94147779, 0.41926275, 0.6832632 ],
       [0.40510405, 0.1340505 , 0.29051973, 0.17273458]])

In [16]:
for qid,q_dict in queries2.items():
    similarity_dict = q_dict['similarity_dict'] = {}
    feature_count = len(q_dict['features'][0])
    for f1_index in range(feature_count):
        for f2_index in range(f1_index + 1, feature_count):
            key = get_sim_key(f1_index, f2_index)
            similarity_dict[key] = kendalls_tau(f1_index, f2_index, q_dict['features'])

In [17]:
for qid,q_dict in queries2.items():
    n = len(q_dict['features'][0])

    q_dict['feature_scores'] = []
    for f_index in range(feature_count):
        c_feature_list = [
            (
                feature_list[f_index],
                score
            ) for feature_list, score in zip(q_dict['features'], q_dict['scores'])
        ]
        feature_sorted_list = [second for _, second in sorted(c_feature_list, key=lambda x: x[0])]
        ndcg_score = ndcg_at_k(feature_sorted_list)
        feature_sorted_list.reverse()
        ndcg_score2 = ndcg_at_k(feature_sorted_list)
        q_dict['feature_scores'].append(max(ndcg_score,ndcg_score2))


In [19]:
size1 = 50
size2 = 25
q_list = list(queries2.items())

train_queries = dict(q_list[:size1])
test_queries = dict(q_list[size1:size1 + size2])

In [20]:
feature_scores = None
similarity_dict = None

for qid, q_dict in train_queries.items():
    if feature_scores is None:
        feature_scores = copy.deepcopy(q_dict['feature_scores'])
        similarity_dict = copy.deepcopy(q_dict['similarity_dict'])
    else:
        feature_scores = [i + j for i, j in zip(feature_scores, q_dict['feature_scores'])]
        for key, score in similarity_dict.items():
            similarity_dict[key] = score + q_dict['similarity_dict'][key]

feature_scores = [score/size1 for score in feature_scores]
similarity_dict = {key: score/size1 for key, score in similarity_dict.items()}
features_ordered = GAS(feature_scores, similarity_dict)

19 0.5535662359743142
6 0.5535662359743142
5 0.5535662359743142
4 0.5523298723379506
18 0.5462383935264336
17 0.5664910968908279
2 0.5377993904053565
16 0.36168211673876977
9 0.5608158411803958
15 0.27588818955505623
0 0.18285651076666448
12 0.020630412381092367
14 -0.21765881150305547
8 -0.13577405780950308
1 -0.31572788232191606
3 -0.5593067386169553
10 -0.8262751733749241
13 -0.8744934440867447
7 -1.3262980661234671


In [129]:
print(features_ordered)

[19, 6, 5, 4, 18, 17, 2, 16, 9, 15, 0, 12, 14, 8, 1, 3, 10, 13, 7, 11]


In [21]:
"""
for qid, q_dict in queries2.items():
    features_ordered = GAS(q_dict['feature_scores'], q_dict['similarity_dict'])
    q_dict['feature_order'] = features_ordered
"""

"\nfor qid, q_dict in queries2.items():\n    features_ordered = GAS(q_dict['feature_scores'], q_dict['similarity_dict'])\n    q_dict['feature_order'] = features_ordered\n"

In [118]:
def prepare_output(queries):
    y = []
    for q_dict in queries.values():
        y += q_dict['scores']
    return y

def prepare_input(queries, features_ordered, num_of_features=4):
    X = []
    for q_dict in queries.values():
        features_list_list = q_dict['features']
        for features_list in features_list_list:
            x_group = []
            for feature_index in features_ordered[:num_of_features]:
                feature_score = features_list[feature_index]
                x_group.append(feature_score)
            X.append(x_group)
    return X
    

In [126]:
y = prepare_output(train_queries)
y_test = prepare_output(test_queries)

y = np.array(y)
y_test = np.array(y_test)

In [15]:
def get_model(num_of_features):
    model = Sequential([
        Input(shape=(num_of_features,)),  # Define the input shape here
        Dense(128, activation='relu'),  # First hidden layer
        Dense(128, activation='relu'),  # Second hidden layer
        Dense(64, activation='relu'),   # Third hidden layer
        Dense(1)  # Output layer
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

In [17]:
class DynamicFeatureDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, x, y, batch_size, shuffle=True):
        self.num_samples, self.num_features = x.shape  # Total number of samples
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(self.num_samples)
        self.x = x
        self.y = y        
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __len__(self):
        return int(np.floor(self.num_samples / self.batch_size))
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __getitem__(self, index):
        batch_X = self.x[index * self.batch_size:(index + 1) * self.batch_size].toarray()
        batch_y = self.y[index * self.batch_size:(index + 1) * self.batch_size]
        return batch_X, batch_y



class DynamicFeatureDataTESTGenerator(tf.keras.utils.Sequence):
    def __init__(self, x, batch_size, shuffle=True):
        self.num_samples, self.num_features = x.shape  # Total number of samples
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(self.num_samples)
        self.x = x
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __len__(self):
        return int(np.floor(self.num_samples / self.batch_size))
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __getitem__(self, index):
        batch_X = self.x[index * self.batch_size:(index + 1) * self.batch_size].toarray()
        return batch_X
    


def pair_data_generator(x, y, batch_size=64):
    ending = x.shape[0]
    index = 0
    while(index<ending):
        e_index = index + batch_size
        if e_index > ending:
            e_index = ending
        x_batch = x[index: e_index].toarray()
        y_batch = y[index: e_index]
        # Yield the batch (features, labels)
        yield x_batch, y_batch
        index += batch_size

def data_generator(x, batch_size=64):
    ending = x.shape[0]
    index = 0
    while(index<ending):
        e_index = index + batch_size
        if e_index > ending:
            e_index = ending
        x_batch = x[index: e_index].toarray()
        # Yield the batch (features, labels)
        yield x_batch
        index += batch_size



# ds_counter = tf.data.Dataset.from_generator(count, args=[25], output_types=tf.int32, output_shapes = (), )
print(type(X_train))
print(X_train.shape)
print(type(X_train[(0,0)]))

aaa=X_train[:1000]
aaa=aaa.toarray()
aaa.shape
print(type(X_train[:10].toarray()))
print(type(y_train[:10]))
print()
for i,j in pair_data_generator(X_train,y_train):
    print(type(i))
    print(type(j))
    break

<class 'scipy.sparse._csr.csr_matrix'>
(1074050, 221)
<class 'numpy.float64'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [18]:
best_score = -9999999999999999
best_model = None
best_num = None
batch_size = 512

for num_of_features in range(100,feature_count):
    feature_indexes = sorted(copy.deepcopy(features_ordered[:num_of_features]))
    training_X = X_train[:, features_ordered[:num_of_features]]
    testing_X = X_test[:, features_ordered[:num_of_features]]
    
    # training_X = np.array(training_X)
    # testing_X = np.array(testing_X)

    model = get_model(num_of_features)

    """
    generated_dataset = tf.data.Dataset.from_generator(
        lambda: pair_data_generator(training_X, y_train, batch_size=64),
        # pair_data_generator(training_X, y_train, batch_size=64),
        output_signature=(
            tf.TensorSpec(shape=(None, num_of_features), dtype=tf.float32),  # Features shape
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)   # Labels shape
        )
    )
    generated_dataset = generated_dataset.shuffle(buffer_size=1000).prefetch(tf.data.AUTOTUNE)
    """

    # model.fit(generated_dataset, epochs=100, steps_per_epoch=training_X.shape[0] // 64)
    # model.fit(generated_dataset, epochs=100, verbose=1)
    generator = DynamicFeatureDataGenerator(training_X, y_train, batch_size=batch_size)
    model.fit(generator, epochs=4, verbose=1)
    # model.fit(training_X, y_train, epochs=100, batch_size=64, verbose=1)

    test_generator = DynamicFeatureDataTESTGenerator(testing_X, batch_size=batch_size)
    y_pred = model.predict(test_generator, verbose=0)
    # y_pred = model.predict(testing_X, verbose=0)
    ndcg_average = 0
    for start, end in qid_limits_test:
        pred_ys = y_pred[start:end]
        real_ys = y_test[start:end]
        scored_docs = list(zip(pred_ys, real_ys))
        sorted_relevances = [data[1] for data in sorted(scored_docs, key=lambda x: x[0], reverse=True)]
        ndcg_score = ndcg_at_k(sorted_relevances)
        ndcg_average += ndcg_score
    ndcg_average /= len(qid_limits_test)
    score = ndcg_average
    print(num_of_features, ndcg_average)

    if score > best_score:
        best_score = score
        best_model = model
        best_num = num_of_features

I0000 00:00:1737256325.414751  140746 service.cc:148] XLA service 0x86c2300 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1737256325.415083  140746 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1737256325.509740  140746 service.cc:148] XLA service 0x85a50c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1737256325.509777  140746 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
I0000 00:00:1737256325.517271  140746 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4232 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoch 1/10


/home/ersel/code/ceng778-termproject/venv/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
2025-01-19 06:12:07.418913: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1737256327.563145  142637 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-01-19 06:12:08.685807: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_362', 448 bytes spill stores, 448 bytes spill loads



   61/16782 ━━━━━━━━━━━━━━━━━━━━ 42s 3ms/step - loss: nan - mae: nan                  

I0000 00:00:1737256329.570436  142637 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 8328/16782 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - loss: nan - mae: nan

KeyboardInterrupt: 

In [132]:
best_model.save('./gas_model.keras')

In [131]:
print(best_num)
print(best_score)
print(features_ordered)
"""
"""

8
0.4823585857797753
[19, 6, 5, 4, 18, 17, 2, 16, 9, 15, 0, 12, 14, 8, 1, 3, 10, 13, 7, 11]


'\n'

In [91]:
from tensorflow.keras.models import load_model
model = load_model('./gas_model.keras')
X = prepare_input(train_queries, features_ordered, 19)
X = np.array(X)
model.predict(X)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  


array([[7.204371],
       [7.898884],
       [8.618757],
       ...,
       [8.654239],
       [8.960614],
       [9.192194]], dtype=float32)

array([21.2268115, 11.       ,  2.       ,  0.       ])

In [ ]:
DQ_INPUT = 1
Q_INPUT = 2
D_INPUT = 3
feature_func_list = [
    (title_query_term_overlap, DQ_INPUT),
    (title_query_jaccard_similarity, DQ_INPUT),
    (title_query_dice_similarity, DQ_INPUT),
    (title_query_position, DQ_INPUT),
    (exact_match_title_query, DQ_INPUT),
    (query_length, Q_INPUT),
    (query_character_length, Q_INPUT),
    (term_overlap, DQ_INPUT),
    (jaccard_similarity, DQ_INPUT),
    (dice_similarity, DQ_INPUT),
    (term_overlap_extra, DQ_INPUT),
    (jaccard_similarity_extra, DQ_INPUT),
    (dice_similarity_extra, DQ_INPUT),
    (document_length, D_INPUT),
    (document_character_length, D_INPUT),
    (average_sentence_length, D_INPUT),
    (unique_word_count, D_INPUT),
    (url_depth, D_INPUT),
    (has_query_parameters, D_INPUT),
]
def get_ordered_features(doc, query):
    return [
        func(doc, query) if DQ_INPUT == input_type 
        else func(doc) if D_INPUT == input_type 
        else func(query) 
        for func, input_type in feature_func_list 
    ]

[1, 2, 3, 5]